# DDS-Edge: train on Kaggle (cleaned 320x240 dataset)

Before running: **Add Input** -> your dataset with `sf_320x240` (images + manifest.csv); do NOT add the original State Farm competition. Session options -> **Accelerator: GPU (P100)**, **Internet: On**.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Bat GPU: Session options -> Accelerator -> GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
%%bash
cd /kaggle/working
[ -d dds-edge ] || git clone https://github.com/ColorOfDreams/dds-edge.git
cd dds-edge && git pull -q
pip install -q timm onnx onnxruntime
pip install -q -e . --no-deps
git log --oneline -1

In [ ]:
# Link the uploaded dataset to where the manifest paths expect it: data/processed/sf_320x240
import os, glob
os.chdir('/kaggle/working/dds-edge')
hits = glob.glob('/kaggle/input/**/manifest.csv', recursive=True)
if not hits:  # dataset uploaded as a .zip that Kaggle did not extract
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'Khong thay manifest.csv hay file .zip: da Add Input dataset sf_320x240 chua?'
    import zipfile
    print('extracting', zips[0])
    zipfile.ZipFile(zips[0]).extractall('/kaggle/working/sf_data')
    hits = glob.glob('/kaggle/working/sf_data/**/manifest.csv', recursive=True)
assert hits, 'Khong thay manifest.csv trong dataset'
src = os.path.dirname(hits[0])
print('dataset at', src, os.listdir(src))
os.makedirs('data/processed', exist_ok=True)
link = 'data/processed/sf_320x240'
if os.path.islink(link):  # stale or broken link from an earlier run
    os.unlink(link)
assert not os.path.exists(link), f'{link} is a real folder, remove it first'
os.symlink(src, link)
import pandas as pd
m = pd.read_csv(link + '/manifest.csv')
print(len(m), 'images;', m.groupby('split')['group'].nunique().to_dict(), 'drivers per split')

In [ ]:
%cd /kaggle/working/dds-edge
# smoke test (1 epoch) - skip once it works
!python -m dds_edge.dl.train --epochs 1 --manifest processed/sf_320x240/manifest.csv --name smoke

In [ ]:
%cd /kaggle/working/dds-edge
# baseline on cleaned data: MobileNetV3-Small, pretrained, aug, grouped6
!python -m dds_edge.dl.train --arch mobilenet_v3_small --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --name mnv3_clean

In [ ]:
%cd /kaggle/working/dds-edge
# other models, same protocol as the baseline. Uncomment ONE line per run.
# !python -m dds_edge.dl.train --arch resnet18 --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --name resnet18_clean
# !python -m dds_edge.dl.train --arch efficientnet_lite0 --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --name effl0_clean
# !python -m dds_edge.dl.train --arch resnet50 --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --name resnet50_clean  # heavy reference

In [ ]:
# package results: download runs.zip from the Output tab (/kaggle/working)
!cd /kaggle/working/dds-edge && zip -qr /kaggle/working/runs.zip runs data/processed/manifest.csv
!ls -lh /kaggle/working/runs.zip